# 04 — Desbalanceamento e limiar

## Objetivo

O desbalanceamento deve ser tratado por pesos, SMOTENC ou por uma decisão de
limiar?

Comparamos somente as três estratégias já validadas. Nenhum novo resampler é
adicionado.

In [1]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import json
import time
import pandas as pd
from imblearn.over_sampling import SMOTENC
from sklearn.utils.class_weight import compute_sample_weight

from src.auxiliares import (
    COLUNAS_NOMINAIS,
    PARAMETROS_REFERENCIA,
    avaliar_probabilidades,
    carregar_base_preparada,
    criar_modelo_gradiente,
    metricas_limiares,
    separar_dados,
)
from src.visual_utils import grafico_metricas_por_limiar

dados = carregar_base_preparada(RAIZ)
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = separar_dados(dados)
caminho_parametros = RAIZ / "models" / "parametros_gradient_boosting.json"
parametros = json.loads(caminho_parametros.read_text()) if caminho_parametros.exists() else PARAMETROS_REFERENCIA.copy()

## Como se comporta o modelo original?

In [2]:
modelo_original = criar_modelo_gradiente(parametros)
inicio = time.perf_counter()
modelo_original.fit(X_treino, y_treino)
tempo_original = time.perf_counter() - inicio
probabilidades_originais = modelo_original.predict_proba(X_validacao)[:, 1]

## Pesos aumentam o alcance da classe positiva?

In [3]:
pesos = compute_sample_weight(class_weight="balanced", y=y_treino)
modelo_pesos = criar_modelo_gradiente(parametros)
inicio = time.perf_counter()
modelo_pesos.fit(X_treino, y_treino, modelo__sample_weight=pesos)
tempo_pesos = time.perf_counter() - inicio
probabilidades_pesos = modelo_pesos.predict_proba(X_validacao)[:, 1]

## O SMOTENC melhora o ranking?

In [4]:
indices_nominais = [X_treino.columns.get_loc(coluna) for coluna in COLUNAS_NOMINAIS]
smote = SMOTENC(categorical_features=indices_nominais, random_state=42)
X_treino_smote, y_treino_smote = smote.fit_resample(X_treino, y_treino)
modelo_smote = criar_modelo_gradiente(parametros)
inicio = time.perf_counter()
modelo_smote.fit(X_treino_smote, y_treino_smote)
tempo_smote = time.perf_counter() - inicio
probabilidades_smote = modelo_smote.predict_proba(X_validacao)[:, 1]

## Qual estratégia preserva melhor a AP / PR-AUC?

In [5]:
resultados_balanceamento = pd.DataFrame([
    avaliar_probabilidades("Original", y_validacao, probabilidades_originais, tempo_treino=tempo_original),
    avaliar_probabilidades("Pesos", y_validacao, probabilidades_pesos, tempo_treino=tempo_pesos),
    avaliar_probabilidades("SMOTENC", y_validacao, probabilidades_smote, tempo_treino=tempo_smote),
]).sort_values("pr_auc", ascending=False)

print("Antes do SMOTENC:", y_treino.value_counts().sort_index().to_dict())
print("Depois do SMOTENC:", pd.Series(y_treino_smote).value_counts().sort_index().to_dict())
resultados_balanceamento

Antes do SMOTENC: {0: 14018, 1: 3982}
Depois do SMOTENC: {0: 14018, 1: 14018}


,modelo,limiar,precision,recall,f1,pr_auc,vn,fp,fn,vp,tempo_treino_s
1,Pesos,0.5,0.474478,0.616428,0.536218,0.553387,3767,906,509,818,30.591683
0,Original,0.5,0.682720,0.363225,0.474176,0.552929,4449,224,845,482,30.798546
2,SMOTENC,0.5,0.505439,0.525245,0.515152,0.529498,3991,682,630,697,51.085797


In [6]:
ap_original = resultados_balanceamento.query("modelo == 'Original'")["pr_auc"].iloc[0]
ap_alternativas = resultados_balanceamento.query("modelo != 'Original'")["pr_auc"]
ganho_maximo = ap_alternativas.max() - ap_original
estrategia_escolhida = "Original" if ganho_maximo <= 0.005 else "Reavaliar"

pd.Series({
    "estrategia_escolhida": estrategia_escolhida,
    "maior_ganho_alternativo_em_ap": ganho_maximo,
    "criterio_minimo": 0.005,
}).to_frame("resultado")

,resultado
estrategia_escolhida,Original
maior_ganho_alternativo_em_ap,0.000458
criterio_minimo,0.005


Pesos elevam Recall, mas também os falsos positivos. Um ganho de AP inferior a
0,005 não compensa essa mudança neste projeto. SMOTENC não precisa vencer para
ensinar: balancear classes não garante ranking melhor.

## O que muda quando alteramos o limiar?

In [7]:
resultados_limiares = metricas_limiares(
    y_validacao,
    probabilidades_originais,
    [0.50, 0.40, 0.30, 0.27],
)
resultados_limiares

,limiar,precision,recall,f1,pr_auc
0,0.50,0.682720,0.363225,0.474176,0.552929
1,0.40,0.628855,0.430294,0.510962,0.552929
2,0.30,0.548714,0.530520,0.539464,0.552929
3,0.27,0.519916,0.560663,0.539521,0.552929


In [8]:
fig = grafico_metricas_por_limiar(resultados_limiares)
fig.show()

## Resultado

O modelo original permanece como solução final. O limiar 0,27 é uma escolha
ilustrativa, definida na prova técnica para aumentar Recall com perda de
Precision. Não há custo financeiro disponível para otimizá-lo.